<a href="https://colab.research.google.com/github/jeolin/BCCE_Experiment/blob/main/randomized_spectral_data_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Research on Copper Sulfate Absorbance Spectrum

Copper(II) sulfate solutions appear blue because they absorb light in the red-orange region of the visible spectrum and transmit blue-green light. The hydrated copper(II) ion, [Cu(H₂O)₆]²⁺, exhibits a characteristic broad absorption band due to d-d electronic transitions.

*   **Expected Shape**: The spectrum typically shows a broad absorption peak.
*   **Lambda Max (Absorbance Maximum)**: For aqueous copper(II) sulfate, the absorption maximum ($\lambda_{max}$) is generally found in the range of **750 nm to 820 nm**, corresponding to the lowest energy d-d transition. The exact position can vary slightly with concentration and solvent.

I will now generate a synthetic spectrum reflecting these properties.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
import random

# --- Define the fixed, broad wavelength range for underlying data ---
wavelengths_data = np.linspace(200, 1200, 1000)

# Generate lambda_max once when the notebook starts
initial_lambda_max = random.randint(795, 820)

def plot_copper_sulfate_spectrum(
    molar_absorptivity_input=1.0,
    min_wavelength_plot=350,
    max_wavelength_plot=1100,
    min_absorbance_plot=0.0,
    max_absorbance_plot=1.0,
    pathlength_input=1.0,
    concentration_input=1.0,
    marker_wavelength_input=350,
    fixed_lambda_max=None # Pass the fixed lambda_max
):
    lambda_max = fixed_lambda_max # Use the pre-generated fixed value

    # Validate and set molar_absorptivity (epsilon)
    try:
        molar_absorptivity = float(molar_absorptivity_input) if molar_absorptivity_input else 1.0
        if molar_absorptivity <= 0:
            molar_absorptivity = 1.0
    except ValueError:
        molar_absorptivity = 1.0

    # Validate and set pathlength
    try:
        pathlength = float(pathlength_input) if pathlength_input else 1.0
        if pathlength <= 0:
            pathlength = 1.0
    except ValueError:
        pathlength = 1.0

    # Validate and set concentration
    try:
        concentration = float(concentration_input) if concentration_input else 1.0
        if concentration <= 0:
            concentration = 1.0
    except ValueError:
        concentration = 1.0

    # Validate plot limits
    if not isinstance(min_wavelength_plot, (int, float)) or not isinstance(max_wavelength_plot, (int, float)):
        min_wavelength_plot = 350
        max_wavelength_plot = 1100
    if min_wavelength_plot >= max_wavelength_plot:
        min_wavelength_plot = 350
        max_wavelength_plot = 1100

    # Validate y-axis plot limits
    if not isinstance(min_absorbance_plot, (int, float)) or not isinstance(max_absorbance_plot, (int, float)):
        min_absorbance_plot = 0.0
        max_absorbance_plot = 1.0
    if min_absorbance_plot >= max_absorbance_plot:
        min_absorbance_plot = 0.0
        max_absorbance_plot = 1.0

    # Validate marker_wavelength_input
    try:
        marker_wavelength = int(marker_wavelength_input) if marker_wavelength_input else 350
        marker_wavelength = max(min(marker_wavelength, 1200), 200)
    except ValueError:
        marker_wavelength = 350

    # Ensure plot limits are within the data generation range
    min_wavelength_plot = max(min_wavelength_plot, min(wavelengths_data))
    max_wavelength_plot = min(max_wavelength_plot, max(wavelengths_data))

    # Generate a synthetic absorbance spectrum using a Gaussian-like distribution
    bandwidth = 70
    gaussian_shape = np.exp(-(wavelengths_data - lambda_max)**2 / (2 * bandwidth**2))
    absorbance_data = molar_absorptivity * pathlength * concentration * gaussian_shape
    absorbance_data += 0.05 * molar_absorptivity * pathlength * concentration

    # Create the plot
    plt.figure(figsize=(5.0, 3.0))
    plt.plot(wavelengths_data, absorbance_data, color='blue')
    plt.title(rf'Absorbance Spectrum of Copper Sulfate (Curve $\lambda_{{max}}$: {lambda_max} nm)')
    plt.ylabel('Absorbance')

    # Mark the user-controlled marker wavelength on the plot
    plt.axvline(x=marker_wavelength, color='red', linestyle='--', label=rf'Marker $\lambda$ = {marker_wavelength} nm')
    text_y_pos = min(max(absorbance_data), max_absorbance_plot * 0.8)
    if text_y_pos > min_absorbance_plot and text_y_pos < max_absorbance_plot:
        plt.text(marker_wavelength + 10, text_y_pos, rf'Marker $\lambda$ = {marker_wavelength} nm', color='red')

    # Apply the user-defined x-axis and y-axis limits
    plt.xlim(min_wavelength_plot, max_wavelength_plot)
    plt.ylim(min_absorbance_plot, max_absorbance_plot)

    plt.grid(True, linestyle='--', alpha=0.7)

# Create interactive widgets
molar_absorptivity_widget = widgets.FloatText(
    value=1.0,
    min=0.01,
    description='Molar Absorptivity (ε):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

pathlength_widget = widgets.FloatText(
    value=1.0,
    min=0.1,
    description='Pathlength (cm):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

concentration_widget = widgets.FloatText(
    value=1.0,
    min=0.001,
    description='Concentration (M):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

marker_wavelength_widget = widgets.IntSlider(
    value=350,
    min=200,
    max=1200,
    step=1,
    description='Marker Wavelength (nm):',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

marker_wavelength_text = widgets.IntText(
    value=350,
    description='(Enter Value):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

widgets.jslink((marker_wavelength_widget, 'value'), (marker_wavelength_text, 'value'))

min_wavelength_widget = widgets.IntText(
    value=350,
    min=200,
    description='Min Wavelength (nm):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

max_wavelength_widget = widgets.IntText(
    value=1100,
    max=1200,
    description='Max Wavelength (nm):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

min_absorbance_widget = widgets.FloatText(
    value=0.0,
    description='Min Absorbance:',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

max_absorbance_widget_y = widgets.FloatText(
    value=1.0,
    description='Max Absorbance:',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

# Group widgets into sections
curve_shape_widgets = widgets.VBox([
    widgets.HTML('<b>Curve Shape Parameters</b>'),
    molar_absorptivity_widget,
    pathlength_widget,
    concentration_widget
])

plot_display_widgets = widgets.VBox([
    widgets.HTML('<b>Plot Display Parameters</b>'),
    widgets.HBox([marker_wavelength_widget, marker_wavelength_text]),
    min_wavelength_widget,
    max_wavelength_widget,
    min_absorbance_widget,
    max_absorbance_widget_y
])

# Combine the control groups into a horizontal box
ui = widgets.HBox([curve_shape_widgets, plot_display_widgets])

# Use widgets.interactive_output to link the widgets to the plotting function
plot_output = widgets.interactive_output(
    plot_copper_sulfate_spectrum,
    {
        'molar_absorptivity_input': molar_absorptivity_widget,
        'min_wavelength_plot': min_wavelength_widget,
        'max_wavelength_plot': max_wavelength_widget,
        'min_absorbance_plot': min_absorbance_widget,
        'max_absorbance_plot': max_absorbance_widget_y,
        'pathlength_input': pathlength_widget,
        'concentration_input': concentration_widget,
        'marker_wavelength_input': marker_wavelength_widget,
        'fixed_lambda_max': widgets.fixed(initial_lambda_max) # Pass the fixed lambda_max
    }
)

# Display the UI (controls) and then the Output widget (plot)
display(ui, plot_output)

Output()

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
from scipy.stats import linregress
import random

def generate_standard_curve(
    selected_wavelength=350,
    num_samples=5,
    min_conc=0.0,
    max_conc=1.0,
    peak_molar_absorptivity_val=1.0, # From previous widget
    pathlength_val=1.0,              # From previous widget
    fixed_lambda_max_val=800         # From previous global variable
):
    # Validate inputs
    if num_samples < 2:
        num_samples = 2
    if min_conc >= max_conc:
        min_conc = 0.0
        max_conc = 1.0
    if max_conc <= 0:
        max_conc = 1.0

    # Generate ideal concentrations and absorbances
    concentrations_ideal = np.linspace(min_conc, max_conc, num_samples)

    # Calculate the Gaussian factor at the selected wavelength using the fixed_lambda_max
    bandwidth = 70  # Same bandwidth as used in the spectrum generation
    gaussian_factor = np.exp(-(selected_wavelength - fixed_lambda_max_val)**2 / (2 * bandwidth**2))

    # Calculate effective molar absorptivity at the selected wavelength
    # The baseline offset (0.05 * peak_molar_absorptivity_val) is also applied proportionally to concentration
    effective_molar_absorptivity_at_wavelength = (peak_molar_absorptivity_val * gaussian_factor) + (0.05 * peak_molar_absorptivity_val)

    # Calculate ideal absorbances using Beer-Lambert Law: A = εbc
    absorbances_ideal = effective_molar_absorptivity_at_wavelength * pathlength_val * concentrations_ideal

    # --- Add (0,0) data point ---
    concentrations = np.insert(concentrations_ideal, 0, 0.0)
    absorbances_with_zero = np.insert(absorbances_ideal, 0, 0.0)

    # --- Iteratively add variability to non-zero data points until R^2 is in desired range ---
    r_squared = 0.0
    attempts = 0
    max_attempts = 500 # Max attempts to find suitable R^2
    target_r2_min = 0.980
    target_r2_max = 0.998

    # Determine max ideal absorbance for scaling noise, ensuring a minimum base for noise calculation
    max_val_for_noise_base = np.max(absorbances_ideal) if len(absorbances_ideal) > 0 else 0.0
    max_val_for_noise = max(max_val_for_noise_base, 0.01) # Use 0.01 as a floor if max_val_for_noise_base is 0.

    absorbances_noisy = np.copy(absorbances_with_zero) # Initialize outside loop for scope
    slope, intercept = 0, 0 # Initialize for scope

    while not (target_r2_min <= r_squared <= target_r2_max) and attempts < max_attempts:
        current_absorbances_noisy = np.copy(absorbances_with_zero)

        # Vary noise level factor for each attempt to hit the R^2 target
        # Expanded the range from 0.005-0.012 to 0.005-0.05 for more variability.
        noise_level_factor = random.uniform(0.005, 0.05)

        for i in range(1, len(current_absorbances_noisy)): # Skip the (0,0) point (index 0)
            noise_std = max_val_for_noise * noise_level_factor
            noise = np.random.normal(0, noise_std)
            current_absorbances_noisy[i] = max(0, absorbances_with_zero[i] + noise)

        # Check if there's enough variation for linear regression
        # If concentrations or absorbances_noisy are all the same, linregress might fail or give R^2=NaN/1.0
        if len(np.unique(concentrations)) < 2 or len(np.unique(current_absorbances_noisy)) < 2:
            r_squared = 0.0 # Force loop to continue if data is flat
        else:
            slope, intercept, r_value, p_value, stderr = linregress(concentrations, current_absorbances_noisy)
            r_squared = r_value**2

        # If a suitable R^2 is found, store the noisy data and break
        if target_r2_min <= r_squared <= target_r2_max:
            absorbances_noisy = current_absorbances_noisy
            break

        attempts += 1

    # If max_attempts reached and R^2 is still not in range, use the last generated noisy data
    # and its corresponding regression parameters. Recalculate slope, intercept, r_squared for consistency.
    if not (target_r2_min <= r_squared <= target_r2_max):
        slope, intercept, r_value, p_value, stderr = linregress(concentrations, absorbances_noisy)
        r_squared = r_value**2

    # Create the plot
    plt.figure(figsize=(5.0, 3.0))

    # Plot noisy data points
    plt.plot(concentrations, absorbances_noisy, 'o', color='green', label='Data Points')

    # Plot regression line
    plt.plot(concentrations, slope * concentrations + intercept, color='red', linestyle='--', label='Linear Regression')

    plt.title(rf'Standard Curve at {selected_wavelength} nm ($\lambda_{{max}}$: {fixed_lambda_max_val} nm)')
    plt.xlabel('Concentration (M)')
    plt.ylabel('Absorbance')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.ylim(bottom=0) # Ensure y-axis starts at 0
    plt.xlim(left=0)  # Ensure x-axis starts at 0

    # --- Display regression equation and correlation coefficient ---
    eq_text = f'y = {slope:.3f}x + {intercept:.3f}'
    r2_text = f'R$^2$ = {r_squared:.3f}'

    # Position text based on plot limits
    x_range = plt.xlim()[1] - plt.xlim()[0]
    y_range = plt.ylim()[1] - plt.ylim()[0]
    plt.text(plt.xlim()[0] + 0.05 * x_range, plt.ylim()[1] - 0.1 * y_range, eq_text, color='red', fontsize=10)
    plt.text(plt.xlim()[0] + 0.05 * x_range, plt.ylim()[1] - 0.18 * y_range, r2_text, color='red', fontsize=10)

    plt.legend()
    plt.show()

# Create widgets for the standard curve
sc_wavelength_widget = widgets.IntSlider(
    value=marker_wavelength_widget.value, # Default to the marker wavelength from the first plot
    min=200,
    max=1200,
    step=1,
    description='Wavelength (nm):',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

sc_num_samples_widget = widgets.IntSlider(
    value=5,
    min=2,
    max=20,
    step=1,
    description='Number of Samples:',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

sc_min_concentration_widget = widgets.FloatText(
    value=0.0,
    min=0.0,
    description='Min Conc. (M):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

sc_max_concentration_widget = widgets.FloatText(
    value=1.0,
    min=0.01,
    description='Max Conc. (M):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

# Group standard curve widgets
standard_curve_widgets = widgets.VBox([
    widgets.HTML('<b>Standard Curve Parameters</b>'),
    sc_wavelength_widget,
    sc_num_samples_widget,
    sc_min_concentration_widget,
    sc_max_concentration_widget
])

# Link to the generate_standard_curve function
standard_curve_output = widgets.interactive_output(
    generate_standard_curve,
    {
        'selected_wavelength': sc_wavelength_widget,
        'num_samples': sc_num_samples_widget,
        'min_conc': sc_min_concentration_widget,
        'max_conc': sc_max_concentration_widget,
        'peak_molar_absorptivity_val': widgets.fixed(molar_absorptivity_widget.value),
        'pathlength_val': widgets.fixed(pathlength_widget.value),
        'fixed_lambda_max_val': widgets.fixed(initial_lambda_max)
    }
)

# Display the standard curve UI and output
display(standard_curve_widgets, standard_curve_output)

Output()